*[Cell 1]*

# Chapter 7: Streaming in and out of your Delta Lake

draft: 3

*[Cell 2]*

# Notebook Intro and Setup

This notebook should run end to end with the "run all" mode.

We encourage exploration and breaking stuff to get the most from it.

In [11]:
# [Cell 3]
# Databricks Free Edition: Spark runs on serverless via Databricks Connect (see dldg.py at the repo root)
from dldg import spark, DATASETS, WORK, set_conf, tree, rm, upload_dir
CH = f"{WORK}/ch07"  # replaces the Docker image's /opt/spark/work-dir

# Remove any old checkpoints and output tables
rm(CH)

# Set some Spark configuration options
set_conf("spark.sql.adaptive.enabled", "false")
set_conf("spark.databricks.delta.schema.autoMerge.enabled","true")

# set COVID dataset path
covid_data_path = f"{DATASETS}/COVID-19_NYT"

# Kill any other active streams
for stream in spark.streams.active:
    stream.stop()

# Serverless only supports trigger(availableNow=True) / trigger(once=True): each stream below processes
# the data available now and then stops, instead of running continuously.

Skipped spark.sql.adaptive.enabled=false: not settable on serverless compute
Skipped spark.databricks.delta.schema.autoMerge.enabled=true: not settable on serverless compute


*[Cell 4]*

# Delta `readStream` example 

In [12]:
# [Cell 5]
streamingDeltaDf = (
    spark
    .readStream
    .format("delta")
    .option("ignoreDeletes", "true")
    .load(covid_data_path)
    )

*[Cell 6]*

# Delta `writeStream` example

In [13]:
# [Cell 7]
(streamingDeltaDf
.writeStream
.format("delta")
.outputMode("append")
.option("checkpointLocation", f"{CH}/delta/ckpt/ws1/")
.trigger(availableNow=True)
.start(f"{CH}/delta/covid/")
.awaitTermination()
);

In [ ]:
# [Cell 8]
# No sleep needed: the availableNow streams above run to completion before returning

*[Cell 9]*

# Delta chained `readStream` to `writeStream`

In [14]:
# [Cell 10]
# Check the count of the table
(spark
.read
.format("delta")
.load(f"{CH}/delta/covid/")
).count()

1111930

In [15]:
# [Cell 11]
from pyspark.sql.functions import col

(spark
.readStream
.format("delta")
.load(f"{CH}/delta/covid/")
# in the book this says '<other transformation logic>'
.filter(col("deaths") > 0)
.writeStream
.format("delta")
.outputMode("append")
.option("checkpointLocation", f"{CH}/delta/ckpt/ws2/")
.trigger(availableNow=True)
.start(f"{CH}/delta/covid_deaths/")
.awaitTermination()
);

In [ ]:
# [Cell 12]
# No sleep needed: the availableNow streams above run to completion before returning

*[Cell 13]*

# Setting `ignoreDeletes`

In [7]:
# [Cell 14]
(
    spark
    .readStream
    .format("delta")
    .option("ignoreDeletes", "true")
    .load(f"{CH}/delta/covid_deaths/")
    );

*[Cell 15]*

# Setting `ignoreChanges`

In [ ]:
# [Cell 16]
streamingDeltaDf = (
    spark
    .readStream
    .format("delta")
    .option("ignoreChanges", "true")
    .load(f"{CH}/delta/covid_deaths/")
    );

*[Cell 17]*

# Specify the `startingVersion`

In [16]:
# [Cell 18]
streamingDeltaDf = (spark
.readStream
.format("delta")
.option("startingVersion", "5")
.load(f"{CH}/delta/covid_deaths/")
);

*[Cell 19]*

# Specify the `startingTimestamp`

In [17]:
# [Cell 20]
streamingDeltaDf = (spark
.readStream
.format("delta")
.option("startingTimestamp", "2023-04-18")
.load(f"{CH}/delta/covid_deaths/")
);

*[Cell 21]*

# Setting `eventTimeOrder` with a watermark

In [18]:
# [Cell 22]
from pyspark.sql.functions import col, current_timestamp

(spark
.readStream
.format("delta")
.load(f"{CH}/delta/covid/")
.withColumn("event_time", current_timestamp())
.writeStream
.format("delta")
.outputMode("append")
.option("checkpointLocation", f"{CH}/delta/ckpt/ws3/")
.trigger(availableNow=True)
.start(f"{CH}/delta/covid_deaths_with_ts/")
.awaitTermination()
);

In [ ]:
# [Cell 23]
# No sleep needed: the availableNow streams above run to completion before returning

In [19]:
# [Cell 24]
streamingDeltaDf = (spark
.readStream
.format("delta")
.option("withEventTimeOrder", "true")
.load(f"{CH}/delta/covid_deaths_with_ts/")
.withWatermark("event_time", "10 seconds")
.writeStream
.format("delta")
.outputMode("append")
.option("checkpointLocation", f"{CH}/delta/ckpt/ws4/")
.trigger(availableNow=True)
.start(f"{CH}/delta/covid_deaths_with_ts2/")
.awaitTermination()
)

*[Cell 25]*

# Idempotent Fanout

Example of a `forEachBatch` function using `txnVersion` options for idempotency

In [20]:
# [Cell 26]
from datetime import datetime, timedelta

# Drop leftovers from earlier runs so the table history starts at version 0 (later cells use fixed versions)
spark.sql("DROP TABLE IF EXISTS workspace.dldg.covid_table")
spark.sql("DROP TABLE IF EXISTS workspace.dldg.covid_table2")

# Create a new table, enable change data feed, get some change records going
(spark
.read
.format("delta")
.load(f"{CH}/delta/covid/")
.write
.format("delta")
.mode("overwrite")
.saveAsTable("workspace.dldg.covid_table")
)
spark.sql("ALTER TABLE workspace.dldg.covid_table SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

start_ts = datetime.now().strftime("%Y-%m-%dT%H:%M:%S")

for i in range(10):
    (spark
    .read
    .format("delta")
    .load(covid_data_path)
    .sample(0.05)
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("workspace.dldg.covid_table")
    );
    if i == 7:
        end_ts = datetime.now().strftime("%Y-%m-%dT%H:%M:%S")

# We'll need a second table too
(spark
.read
.format("delta")
.load(f"{CH}/delta/covid/")
.sample(0.01)
.drop_duplicates(["fips", "date", "county", "state"])
.write
.format("delta")
.mode("overwrite")
.saveAsTable("workspace.dldg.covid_table2")
)

In [21]:
# [Cell 27]
# No sleep needed: the availableNow streams above run to completion before returning

In [ ]:
# [Cell 28]
# NOTE: foreachBatch runs this Python function on Databricks. Over Databricks Connect on Free Edition it fails
# with ISOLATION_STARTUP_FAILURE.SANDBOX_STARTUP (Python UDFs fail the same way). Run this cell from a
# Databricks notebook in the browser instead (dldg.py works there too).
app_id = "delta_idempotency_example" # A unique string used as an application ID.

def writeToDeltaLakeTableIdempotent(batch_df, batch_id):
    # location 1
    (batch_df
    .write
    .format("delta")
    .option("txnVersion", batch_id)
    .option("txnAppId", app_id)
    .save(f"{CH}/delta/idempotent_location_1/")
    )
    # location 2
    (batch_df
    .write
    .format("delta")
    .option("txnVersion", batch_id)
    .option("txnAppId", app_id)
    .save(f"{CH}/delta/idempotent_location_2/")
    )

    
changesStream = ( # Streaming dataframe with CDC records
    spark
    .readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 1)
    .table("workspace.dldg.covid_table")
    )

# Write the output of a streaming aggregation query into Delta table
(changesStream
.writeStream
.format("delta")
.queryName("A Pipeline")
.option("checkpointLocation", f"{CH}/delta/ckpt/ws5/")  # serverless requires an explicit checkpoint
.foreachBatch(writeToDeltaLakeTableIdempotent)
.outputMode("update")
.trigger(availableNow=True)
.start()
.awaitTermination()
);

*[Cell 29]*

# Streaming Upsert

Using the Delta Lake `mergeBuilder` to create an upsert `forEachBatch` function and apply to a `writeStream`

In [ ]:
# [Cell 30]
# spark.table("workspace.dldg.covid_table").printSchema()
changesStream.printSchema()

In [ ]:
# [Cell 31]
# NOTE: foreachBatch runs this Python function on Databricks. Over Databricks Connect on Free Edition it fails
# with ISOLATION_STARTUP_FAILURE.SANDBOX_STARTUP (Python UDFs fail the same way). Run this cell from a
# Databricks notebook in the browser instead (dldg.py works there too).
from delta.tables import *

changesStream = ( # Streaming dataframe with CDC records
    spark
    .readStream
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 1)
    .table("workspace.dldg.covid_table")
    )

# Function to upsert microBatchDf into Delta table using merge
def upsertToDelta(microBatchDf, batchId):
    """Use Delta APIs to handle merge logic into table"""
    deltaTable = DeltaTable.forName(spark, "workspace.dldg.covid_table2") # Target table

    # The book checks for 'DELETE'; CDF change types are lowercase ('delete')
    deltaTable.alias("dt") \
    .merge(
        # CDF emits update_preimage + update_postimage rows per update; keep only the new values
        source = microBatchDf.filter("_change_type != 'update_preimage'").alias("sdf").drop_duplicates(["fips", "date", "county", "state"]),
        condition = "sdf.fips = dt.fips and sdf.date = dt.date and sdf.county = dt.county and sdf.state = dt.state"
        ) \
    .whenMatchedDelete(condition = "sdf._change_type = 'delete'") \
    .whenMatchedUpdate(set = {
        "fips": "sdf.fips",
        "date": "sdf.date",
        "cases": "sdf.cases",
        "deaths": "sdf.deaths"
        }) \
    .whenNotMatchedInsert(values = {
        "fips": "sdf.fips",
        "date": "sdf.date",
        "cases": "sdf.cases",
        "deaths": "sdf.deaths"
        }) \
    .execute()

# Write the output of a streaming aggregation query into Delta table
(changesStream
.writeStream
.format("delta")
.queryName("New Pipeline")
.option("checkpointLocation", f"{CH}/delta/ckpt/ws6/")  # serverless requires an explicit checkpoint
.foreachBatch(upsertToDelta)
.outputMode("update")
.trigger(availableNow=True)
.start()
.awaitTermination()
)

*[Cell 32]*

# Delta Live Tables (DLT)

A syntactical example, you'll have to go try out something like this on Databricks

*[Cell 33]*

```
import dlt

@dlt.table
def autoloader_dlt_bronze():
    return (
        spark
        .readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .load("<data path>")
    )

@dlt.table
def delta_dlt_silver():
    return (
        dlt
        .read_stream("autoloader_dlt_bronze")
        …
        <transformation logic>
        …
    )

@dlt.table
def live_delta_gold():
    return (
        dlt
        .read("delta_dlt_silver")
        …
        <aggregation logic>
        …
    )
```

*[Cell 34]*

# Defining Change Data Feed read boundaries in a batch process

In [ ]:
# [Cell 35]
print(start_ts, end_ts)

# Specify the version as int or long
(spark
  .read
  .format("delta")
  .option("readChangeFeed", "true")
  .option("startingVersion", 1)
  .option("endingVersion", 10)
  .table("workspace.dldg.covid_table")
)

# Specify timestamps as formatted timestamp
(spark
  .read
  .format("delta")
  .option("readChangeFeed", "true")
  .option("startingTimestamp", start_ts)
  .option("endingTimestamp", end_ts)
  .table("workspace.dldg.covid_table")
)
 
# Providing only the startingVersion/timestamp
(spark
  .read
  .format("delta")
  .option("readChangeFeed", "true")
  .option("startingTimestamp", start_ts)
  .table("workspace.dldg.covid_table")
)

# Specifying similarly with a file location
(spark
  .read
  .format("delta")
  .option("readChangeFeed", "true")
  .option("startingTimestamp", start_ts)
  .table("workspace.dldg.covid_table")  # Unity Catalog blocks path access to managed tables, so read by name
)

*[Cell 36]*

# Defining Change Data Feed read boundaries in a streaming process

In [ ]:
# [Cell 37]
# Specifying a starting version
spark.readStream.format("delta") \
  .option("readChangeFeed", "true") \
  .option("startingVersion", 1) \
  .table("workspace.dldg.covid_table")  # Unity Catalog blocks path access to managed tables

# Specifying a starting timestamp
spark.readStream.format("delta") \
  .option("readChangeFeed", "true") \
  .option("startingTimestamp", start_ts) \
  .table("workspace.dldg.covid_table")  # Unity Catalog blocks path access to managed tables

# Not providing either option, i.e., process from beginning
spark.readStream.format("delta") \
  .option("readChangeFeed", "true") \
  .table("workspace.dldg.covid_table")  # Unity Catalog blocks path access to managed tables

*[Cell 38]*

# Example for viewing changes

In [ ]:
# [Cell 39]
for stream in spark.streams.active:
    stream.stop()

In [ ]:
# [Cell 40]
from pyspark.sql.functions import col

spark.sql("""
    update workspace.dldg.covid_table
        SET
            cases = 100,
            deaths = 1
        WHERE
            date='2020-07-08'
            and fips=21025;
    """)


In [ ]:
# [Cell 41]
# The book hard-codes version 12; look up the UPDATE commit instead
update_version = spark.sql("DESCRIBE HISTORY workspace.dldg.covid_table LIMIT 1").first()["version"]

(spark
.read
.format("delta")
.option("readChangeFeed", "true")
.option("startingVersion", update_version)
.table("workspace.dldg.covid_table")
.select(
    col("date"),
    col("fips"),
    col("cases"),
    col("deaths"),
    col("_change_type"),
    col("_commit_version"))
).show()